# LeetCode 2309 — Greatest English Letter in Upper and Lower Case bằng Python

Link đề: https://leetcode.com/problems/greatest-english-letter-in-upper-and-lower-case/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def greatestLetter(self, s: str) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu khái niệm một chữ cái **xuất hiện ở cả hai dạng** (hoa và thường) trong cùng một chuỗi và cách kiểm tra bằng `set`.
2. Dùng được `set`, phép **giao tập hợp** `&`, `str.upper()`/`str.lower()`, `max(..., default=...)`, `ord`/`chr`.
3. Hiểu cách **duyệt theo thứ tự giảm dần** `"ZYX...A"` để dừng sớm ở chữ cái lớn nhất, hoặc dùng `max` trên các ứng viên.
4. Viết 5 cách: kiểm tra `in` trực tiếp, `set` + duyệt `Z→A`, hai mảng boolean 26 phần tử, giao hai tập đã chuẩn hóa, và bitmask 26 bit.
5. Tránh các lỗi: trả chữ thường thay vì chữ hoa, chỉ xét một dạng, nhầm "lớn nhất" theo thứ tự xuất hiện thay vì thứ tự bảng chữ cái, quên trường hợp rỗng.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm chữ cái tiếng Anh (hoa và thường). Tìm **chữ cái lớn nhất** (theo thứ tự bảng chữ cái, `A < B < ... < Z`) sao cho **cả dạng thường lẫn dạng hoa** của nó đều xuất hiện trong `s`. Trả về chữ cái đó dưới dạng **chữ hoa**. Nếu không có, trả về chuỗi rỗng `""`.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"lEeTcOdE"` | `"E"` | `e` và `E` đều có; `O`/`o`: có `O` nhưng không có `o`; chỉ `E` hợp lệ |
| 2 | `"arRAzFif"` | `"R"` | `a`/`A` và `r`/`R` đều có; `R` lớn hơn `A`; `z`, `F`, `i`, `f`: `f` và `F` có cả hai nhưng `F < R` |
| 3 | `"AbCdEfGhIjK"` | `""` | không có chữ cái nào xuất hiện ở cả hai dạng |

Constraints: `1 <= len(s) <= 1000`; `s` chỉ gồm chữ cái tiếng Anh hoa và thường.

In [1]:
examples = [("lEeTcOdE", "E"), ("arRAzFif", "R"), ("AbCdEfGhIjK", "")]
for s, expected in examples:
    both = sorted(c.upper() for c in set(s) if c.islower() and c.upper() in s)
    print(f"s={s!r:14} chữ cái có cả hai dạng: {both} ; mong đợi {expected!r}")

s='lEeTcOdE'     chữ cái có cả hai dạng: ['E'] ; mong đợi 'E'
s='arRAzFif'     chữ cái có cả hai dạng: ['A', 'F', 'R'] ; mong đợi 'R'
s='AbCdEfGhIjK'  chữ cái có cả hai dạng: [] ; mong đợi ''


## 3. Kiến thức nền

### 3.1. Chuỗi, `upper()`, `lower()`, `islower()`, `isupper()`

`"a".upper() == "A"`, `"A".lower() == "a"`. `islower()`/`isupper()` kiểm tra loại chữ cái.

In [2]:
print("a".upper(), "A".lower(), "e".islower(), "E".isupper(), "E".islower())

A a True True False


### 3.2. `set` — tập hợp các phần tử khác nhau

`set(s)` lấy các ký tự **khác nhau** của `s`, không có thứ tự và không lặp. Tra cứu `x in a_set` trung bình `O(1)`.

In [3]:
s = "lEeTcOdE"
seen = set(s)
print(seen)
print("e" in seen, "E" in seen, "O" in seen, "o" in seen)

{'d', 'c', 'O', 'E', 'l', 'T', 'e'}
True True True False


### 3.3. Giao tập hợp `&`

`A & B` là các phần tử **thuộc cả hai** tập. Nếu `A` là các chữ thường đã có và `B` là các chữ hoa đã chuẩn hóa về chữ thường, `A & B` là các chữ cái có cả hai dạng.

In [4]:
lowers = {c for c in "lEeTcOdE" if c.islower()}
uppers = {c.lower() for c in "lEeTcOdE" if c.isupper()}
print(lowers, uppers, lowers & uppers)

{'l', 'd', 'e', 'c'} {'t', 'o', 'e'} {'e'}


### 3.4. `max(..., default=...)` và thứ tự chữ cái

Chữ cái được so sánh theo **mã ký tự**: `"A" < "B" < ... < "Z"` và `"a" < ... < "z"` (chữ thường có mã lớn hơn chữ hoa, nên **không** trộn hoa/thường khi so sánh). Ta chuẩn hóa về một dạng trước khi `max`. `max([], default="")` trả `""` thay vì lỗi.

In [5]:
print("A" < "B", "Z" < "a", "a" < "B")       # cẩn thận: 'a' > 'B' theo mã
print(max(["E", "R"], default=""), repr(max([], default="")))

True True False
R ''


### 3.5. `ord` và `chr`, tạo `"ZYX...A"`

`ord("A") == 65`, `chr(65) == "A"`. `string.ascii_uppercase` là `"ABC...Z"`; đảo bằng `[::-1]` được `"ZYX...A"`.

In [6]:
import string
print(ord("A"), ord("Z"), chr(65), ord("e") - ord("a"))
print(string.ascii_uppercase)
print(string.ascii_uppercase[::-1])

65 90 A 4
ABCDEFGHIJKLMNOPQRSTUVWXYZ
ZYXWVUTSRQPONMLKJIHGFEDCBA


### 3.6. Bitmask: 26 bit cho 26 chữ cái

Một số nguyên có thể là **tập các chữ cái**: bit `k` bật nghĩa là chữ cái thứ `k` có mặt. `mask |= 1 << k` thêm; `mask1 & mask2` giao hai tập; `mask.bit_length() - 1` là chỉ số bit cao nhất (chữ cái lớn nhất).

In [7]:
mask = 0
for ch in "ace":
    mask |= 1 << (ord(ch) - ord("a"))
print(bin(mask), mask.bit_length() - 1, chr(ord("a") + mask.bit_length() - 1))

0b10101 4 e


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2: `s = "arRAzFif"`. Liệt kê từng chữ cái và hỏi: dạng thường có không? dạng hoa có không?

| chữ cái | thường | hoa | có cả hai? |
|---|---|---|---|
| a | có (`a`) | có (`A`) | **có** |
| f | có (`f`) | có (`F`) | **có** |
| i | có | không | không |
| r | có | có (`R`) | **có** |
| z | có | không | không |

Các chữ cái có cả hai dạng: `A`, `F`, `R`. Lớn nhất theo bảng chữ cái là `R`. Đáp án `"R"`.

Vậy: **(1) tìm tập chữ cái có cả hai dạng, (2) lấy lớn nhất, (3) viết hoa**.

In [8]:
s = "arRAzFif"
for letter in "abcdefghijklmnopqrstuvwxyz":
    lower_in = letter in s
    upper_in = letter.upper() in s
    if lower_in or upper_in:
        print(f"{letter}: thường={lower_in!s:5} hoa={upper_in!s:5} cả hai={lower_in and upper_in}")

a: thường=True  hoa=True  cả hai=True
f: thường=True  hoa=True  cả hai=True
i: thường=True  hoa=False cả hai=False
r: thường=True  hoa=True  cả hai=True
z: thường=True  hoa=False cả hai=False


## 5. Cách 1 — Brute Force / baseline: thử từng chữ cái bằng `in` trên chuỗi

Với mỗi chữ hoa `c` từ `Z` xuống `A`, kiểm tra `c in s` và `c.lower() in s` (mỗi `in` quét cả chuỗi). Gặp cái đầu tiên đủ cả hai thì trả về — vì ta duyệt giảm dần nên đó là cái lớn nhất.

In [9]:
def greatest_brute(s):
    for code in range(ord("Z"), ord("A") - 1, -1):
        upper = chr(code)
        lower = upper.lower()
        if upper in s and lower in s:
            return upper
    return ""

print(greatest_brute("lEeTcOdE"), greatest_brute("arRAzFif"), repr(greatest_brute("AbCdEfGhIjK")))

E R ''


In [10]:
from typing import List

class SolutionBrute:
    def greatestLetter(self, s: str) -> str:
        for code in range(ord("Z"), ord("A") - 1, -1):
            upper = chr(code)
            if upper in s and upper.lower() in s:
                return upper
        return ""

print(SolutionBrute().greatestLetter("lEeTcOdE"))
print(SolutionBrute().greatestLetter("arRAzFif"))
print(repr(SolutionBrute().greatestLetter("AbCdEfGhIjK")))

E
R
''


**Độ phức tạp:** `n = len(s)`. Tối đa 26 chữ cái × 2 phép `in` × `O(n)` → **Time `O(26·n) = O(n)`** (hằng số ~52), **Space `O(1)`**.

**Nút thắt:** mỗi `in` trên chuỗi quét lại toàn bộ `s`. Với `n = 1000` vẫn nhanh, nhưng ta có thể dựng **`set` một lần** để mỗi tra cứu `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

- Chỉ **sự có mặt** của từng ký tự là quan trọng, không phải vị trí hay số lần → `set(s)` (hoặc mảng/bitmask) đủ thông tin.
- Có 26 chữ cái; duyệt từ `Z` xuống `A` và trả về **ngay** chữ cái đầu tiên có cả `upper` và `lower` trong tập; đó chắc chắn là chữ cái lớn nhất.
- Cách khác: lấy **giao** của tập chữ thường và tập chữ hoa (đã đổi về thường) rồi `max`.
- Quy tắc: `nếu c.upper() in seen và c.lower() in seen thì c là ứng viên`; trả về ứng viên lớn nhất, viết hoa. Không có ứng viên → `""`.

## 7. Các cách tiếp theo

### Cách 2 — `set` + duyệt `Z → A` (khuyến nghị khi nộp)

In [11]:
import string
from typing import List

class SolutionSet:
    def greatestLetter(self, s: str) -> str:
        seen = set(s)
        for upper in reversed(string.ascii_uppercase):
            if upper in seen and upper.lower() in seen:
                return upper
        return ""

print(SolutionSet().greatestLetter("lEeTcOdE"))
print(SolutionSet().greatestLetter("arRAzFif"))
print(repr(SolutionSet().greatestLetter("AbCdEfGhIjK")))

E
R
''


**Độ phức tạp:** dựng `set` `O(n)`, duyệt 26 chữ cái `O(26)` → Time `O(n)`, Space `O(1)` (tối đa 52 phần tử).

### Cách 3 — Hai mảng boolean 26 phần tử

`lower_seen[i]` / `upper_seen[i]` ghi chữ cái thứ `i` đã xuất hiện ở dạng thường / hoa. Duyệt `i` từ `25` xuống `0`.

In [12]:
from typing import List

class SolutionArrays:
    def greatestLetter(self, s: str) -> str:
        lower_seen = [False] * 26
        upper_seen = [False] * 26
        for ch in s:
            if ch.islower():
                lower_seen[ord(ch) - ord("a")] = True
            else:
                upper_seen[ord(ch) - ord("A")] = True
        for i in range(25, -1, -1):
            if lower_seen[i] and upper_seen[i]:
                return chr(ord("A") + i)
        return ""

print(SolutionArrays().greatestLetter("lEeTcOdE"))
print(SolutionArrays().greatestLetter("arRAzFif"))
print(repr(SolutionArrays().greatestLetter("AbCdEfGhIjK")))

E
R
''


**Độ phức tạp:** Time `O(n + 26)`, Space `O(26) = O(1)`.

### Cách 4 — Giao hai tập đã chuẩn hóa rồi `max`

Tách `s` thành tập chữ thường và tập chữ hoa **đã đổi về chữ thường**; giao hai tập là các chữ cái có cả hai dạng. Lấy `max` (theo bảng chữ cái, cùng dạng thường nên so sánh đúng) rồi `upper()`.

In [13]:
from typing import List

class SolutionIntersect:
    def greatestLetter(self, s: str) -> str:
        lowers = {c for c in s if c.islower()}
        uppers = {c.lower() for c in s if c.isupper()}
        both = lowers & uppers
        return max(both, default="").upper()

print(SolutionIntersect().greatestLetter("lEeTcOdE"))
print(SolutionIntersect().greatestLetter("arRAzFif"))
print(repr(SolutionIntersect().greatestLetter("AbCdEfGhIjK")))

E
R
''


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (hai tập tối đa 26 phần tử). `"".upper()` vẫn là `""` nên trường hợp rỗng đúng.

### Cách 5 — Bitmask 26 bit

Hai số nguyên `low` và `up` biểu diễn hai tập. `both = low & up`; nếu `both == 0` → `""`; ngược lại bit cao nhất `both.bit_length() - 1` là chữ cái lớn nhất.

In [14]:
from typing import List

class SolutionBitmask:
    def greatestLetter(self, s: str) -> str:
        low = up = 0
        for ch in s:
            if ch.islower():
                low |= 1 << (ord(ch) - ord("a"))
            else:
                up |= 1 << (ord(ch) - ord("A"))
        both = low & up
        if both == 0:
            return ""
        return chr(ord("A") + both.bit_length() - 1)

print(SolutionBitmask().greatestLetter("lEeTcOdE"))
print(SolutionBitmask().greatestLetter("arRAzFif"))
print(repr(SolutionBitmask().greatestLetter("AbCdEfGhIjK")))

E
R
''


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (hai số nguyên 26 bit). Rất gọn và nhanh; `bit_length` cho vị trí bit cao nhất tương ứng chữ cái lớn nhất.

## 8. TRACE từng bước

Trace Cách 5 (bitmask) với `s = "arRAzFif"`:

| ký tự | loại | `low` (bit các chữ thường) | `up` (bit các chữ hoa) |
|---|---|---|---|
| `a` | thường | `a` | — |
| `r` | thường | `a, r` | — |
| `R` | hoa | `a, r` | `R` |
| `A` | hoa | `a, r` | `R, A` |
| `z` | thường | `a, r, z` | `R, A` |
| `F` | hoa | `a, r, z` | `R, A, F` |
| `i` | thường | `a, i, r, z` | `R, A, F` |
| `f` | thường | `a, f, i, r, z` | `R, A, F` |

`both = low & up` ứng với `{a, f, r}`; bit cao nhất là `r` → `"R"`.

In [15]:
def trace_bitmask(s):
    def letters(mask):
        return "".join(chr(97 + i) for i in range(26) if mask >> i & 1)
    low = up = 0
    print(f"s={s!r}")
    for ch in s:
        if ch.islower():
            low |= 1 << (ord(ch) - 97)
        else:
            up |= 1 << (ord(ch) - 65)
        print(f"  ch={ch!r}: low={letters(low)!r:14} up={letters(up)!r}")
    both = low & up
    print("  both =", repr(letters(both)))
    result = chr(65 + both.bit_length() - 1) if both else ""
    print("  kết quả:", repr(result))
    return result

trace_bitmask("arRAzFif")
trace_bitmask("AbCdEfGhIjK")

s='arRAzFif'
  ch='a': low='a'            up=''
  ch='r': low='ar'           up=''
  ch='R': low='ar'           up='r'
  ch='A': low='ar'           up='ar'
  ch='z': low='arz'          up='ar'
  ch='F': low='arz'          up='afr'
  ch='i': low='airz'         up='afr'
  ch='f': low='afirz'        up='afr'
  both = 'afr'
  kết quả: 'R'
s='AbCdEfGhIjK'
  ch='A': low=''             up='a'
  ch='b': low='b'            up='a'
  ch='C': low='b'            up='ac'
  ch='d': low='bd'           up='ac'
  ch='E': low='bd'           up='ace'
  ch='f': low='bdf'          up='ace'
  ch='G': low='bdf'          up='aceg'
  ch='h': low='bdfh'         up='aceg'
  ch='I': low='bdfh'         up='acegi'
  ch='j': low='bdfhj'        up='acegi'
  ch='K': low='bdfhj'        up='acegik'
  both = ''
  kết quả: ''


''

In [16]:
def trace_set(s):
    seen = set(s)
    print(f"s={s!r}, tập ký tự đã thấy = {''.join(sorted(seen))!r}")
    for code in range(ord("Z"), ord("A") - 1, -1):
        up = chr(code)
        lo = up.lower()
        ok = up in seen and lo in seen
        if up in seen or lo in seen:
            print(f"  thử {up}: hoa={up in seen!s:5} thường={lo in seen!s:5} -> {ok}")
        if ok:
            return up
    return ""

print("kết quả:", repr(trace_set("lEeTcOdE")))
print("kết quả:", repr(trace_set("arRAzFif")))

s='lEeTcOdE', tập ký tự đã thấy = 'EOTcdel'
  thử T: hoa=True  thường=False -> False
  thử O: hoa=True  thường=False -> False
  thử L: hoa=False thường=True  -> False
  thử E: hoa=True  thường=True  -> True
kết quả: 'E'
s='arRAzFif', tập ký tự đã thấy = 'AFRafirz'
  thử Z: hoa=False thường=True  -> False
  thử R: hoa=True  thường=True  -> True
kết quả: 'R'


## 9. Code nộp LeetCode

In [17]:
import string
from typing import List

class Solution:
    def greatestLetter(self, s: str) -> str:
        seen = set(s)
        for upper in reversed(string.ascii_uppercase):
            if upper in seen and upper.lower() in seen:
                return upper
        return ""

print(Solution().greatestLetter("lEeTcOdE"))
print(Solution().greatestLetter("arRAzFif"))
print(repr(Solution().greatestLetter("AbCdEfGhIjK")))

E
R
''


## 10. Vì sao thuật toán đúng

- **Ứng viên đúng định nghĩa:** chữ cái `L` đủ điều kiện ⇔ `L.upper() ∈ s` và `L.lower() ∈ s`. Với `set(s)`, thành viên `x ∈ set(s)` ⇔ `x` xuất hiện ít nhất một lần trong `s`.
- **Tính lớn nhất:** ta duyệt `Z, Y, ..., A`. **Invariant:** khi đang xét chữ `c`, mọi chữ lớn hơn `c` đã bị loại. Do đó chữ đầu tiên đủ điều kiện là chữ lớn nhất.
- **Chuẩn hóa:** so sánh chỉ thực hiện giữa các chữ cùng dạng (hoa với hoa), nên không bị lệch bởi việc mã chữ thường (`97..122`) lớn hơn mã chữ hoa (`65..90`).
- **Không có kết quả:** nếu không chữ nào đủ điều kiện, vòng lặp kết thúc và trả `""`.
- **Bitmask:** `low & up` giữ đúng những bit có mặt ở cả hai; `bit_length() - 1` là bit cao nhất, tương ứng chữ cái lớn nhất theo thứ tự `a..z`.

## 11. Phân tích Time / Space Complexity

`n = len(s) ≤ 1000`; bảng chữ cái cố định 26.

| Cách | Time | Space |
|---|---|---|
| 1. `in` trên chuỗi mỗi chữ cái | `O(26·n)` | `O(1)` |
| 2. `set` + duyệt `Z→A` | `O(n + 26)` | `O(1)` |
| 3. Hai mảng boolean | `O(n + 26)` | `O(1)` |
| 4. Giao hai tập + `max` | `O(n)` | `O(1)` |
| 5. Bitmask 26 bit | `O(n)` | `O(1)` |

Cận dưới `O(n)` vì phải đọc `s`. Hằng số 26 coi như hằng.

## 12. Bộ test

Ví dụ đề, edge case (một ký tự, chỉ một dạng, toàn bộ cả hai dạng, `Z/z`, `A/a`), phản ví dụ, và test ngẫu nhiên đối chiếu **tất cả cách**.

In [18]:
import random, string

solvers = [SolutionBrute(), SolutionSet(), SolutionArrays(), SolutionIntersect(), SolutionBitmask(), Solution()]

fixed = [
    ("lEeTcOdE", "E"),
    ("arRAzFif", "R"),
    ("AbCdEfGhIjK", ""),
    ("a", ""),                         # một ký tự thường
    ("A", ""),                         # một ký tự hoa
    ("aA", "A"),                       # nhỏ nhất có đáp án
    ("zZ", "Z"),                       # lớn nhất
    ("aAzZ", "Z"),
    ("abcABC", "C"),
    ("abcdefg", ""),                   # chỉ thường
    ("ABCDEFG", ""),                   # chỉ hoa
    ("aBcDeFg", ""),                   # mỗi chữ cái chỉ một dạng
    ("Zaz", "Z"),                      # Z và z đều có; a chỉ một dạng
    ("Aaz", "A"),                      # 'z' lớn nhất nhưng không có 'Z'
]
for s, expected in fixed:
    for sol in solvers:
        assert sol.greatestLetter(s) == expected, (type(sol).__name__, s, sol.greatestLetter(s), expected)
print("test cố định: OK")

def reference(s):
    return max((c.upper() for c in s if c.lower() in s and c.upper() in s), default="")

random.seed(2309)
non_empty = 0
for _ in range(1000):
    n = random.randint(1, 30)
    pool = random.choice([string.ascii_letters, "aAbBcC", "abcxyzXYZ", string.ascii_lowercase[:8] + string.ascii_uppercase[8:16]])
    s = "".join(random.choice(pool) for _ in range(n))
    expected = reference(s)
    non_empty += expected != ""
    for sol in solvers:
        assert sol.greatestLetter(s) == expected, (type(sol).__name__, s)
print("1000 test ngẫu nhiên: OK, số lần đáp án khác rỗng:", non_empty)
assert non_empty > 0

test cố định: OK
1000 test ngẫu nhiên: OK, số lần đáp án khác rỗng: 595


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Trả về chữ thường thay vì chữ hoa

Đề yêu cầu kết quả **chữ hoa**.

In [19]:
def wrong1(s):
    seen = set(s)
    for upper in reversed("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
        if upper in seen and upper.lower() in seen:
            return upper.lower()
    return ""

print("sai :", repr(wrong1("lEeTcOdE")), "(đáng lẽ 'E')")
print("đúng:", repr(Solution().greatestLetter("lEeTcOdE")))

sai : 'e' (đáng lẽ 'E')
đúng: 'E'


### Lỗi 2: Chỉ kiểm tra một dạng

Chữ cái chỉ có dạng hoa (hoặc chỉ thường) **không** thỏa. Nếu chỉ kiểm tra `upper in seen`, `"Z"` bị chọn dù không có `z`.

In [20]:
def wrong2(s):
    seen = set(s)
    for upper in reversed("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
        if upper in seen:                      # thiếu upper.lower() in seen
            return upper
    return ""

print("sai :", repr(wrong2("AbCdEfGhIjK")), "(đáng lẽ '')")
print("đúng:", repr(Solution().greatestLetter("AbCdEfGhIjK")))

sai : 'K' (đáng lẽ '')
đúng: ''


### Lỗi 3: Lấy `max` trên mọi ký tự rồi đổi hoa, không kiểm tra cặp

Mã chữ thường lớn hơn mã chữ hoa, nên `max(set(s))` luôn nghiêng về chữ thường lớn nhất — dù nó **không có cặp** chữ hoa.

In [21]:
def wrong3(s):
    return max(set(s)).upper()

print("sai :", repr(wrong3("Aaz")), "(max chọn 'z' -> 'Z'; đáng lẽ 'A')")
print("đúng:", repr(Solution().greatestLetter("Aaz")))

sai : 'Z' (max chọn 'z' -> 'Z'; đáng lẽ 'A')
đúng: 'A'


### Lỗi 4: Lấy chữ cái xuất hiện **cuối cùng/đầu tiên** thay vì lớn nhất

"Lớn nhất" theo bảng chữ cái, không phải theo vị trí trong chuỗi.

In [22]:
def wrong4(s):
    seen = set(s)
    for ch in s:                                # theo thứ tự xuất hiện, không phải thứ tự chữ cái
        if ch.swapcase() in seen:
            return ch.upper()
    return ""

print("sai :", repr(wrong4("aAzZ")), "(đáng lẽ 'Z')")
print("đúng:", repr(Solution().greatestLetter("aAzZ")))

sai : 'A' (đáng lẽ 'Z')
đúng: 'Z'


### Lỗi 5: `max` trên danh sách rỗng

Không có chữ cái hợp lệ thì `max([])` ném `ValueError`; dùng `default=""`.

In [23]:
def wrong5(s):
    lowers = {c for c in s if c.islower()}
    uppers = {c.lower() for c in s if c.isupper()}
    return max(lowers & uppers).upper()

try:
    wrong5("AbCdEfGhIjK")
except ValueError as e:
    print("sai -> ValueError:", e)
print("đúng:", repr(Solution().greatestLetter("AbCdEfGhIjK")))

sai -> ValueError: max() iterable argument is empty
đúng: ''


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chữ cái xuất hiện ở **cả hai** dạng/tập" → `set` + **giao** `&` hoặc kiểm tra `in` hai lần.
- "Lớn nhất theo bảng chữ cái" với chỉ 26 khả năng → **duyệt ngược từ `Z`** và dừng sớm.
- Bài chỉ cần **sự có mặt** (không cần số lần) → `set`, mảng boolean hoặc bitmask.
- `str.swapcase()` đổi hoa↔thường, tiện để kiểm tra "dạng đối".
- Luôn chuẩn hóa về một dạng (`lower()`/`upper()`) trước khi so sánh thứ tự.
- Nhớ trường hợp **rỗng**: `max(..., default="")`.

## 15. Biến thể của bài

1. Trả về chữ cái **nhỏ nhất** có cả hai dạng.
2. Đếm **số chữ cái** có cả hai dạng (kích thước tập giao).
3. Trả về chữ cái có cả hai dạng xuất hiện **nhiều lần nhất**.
4. LeetCode 1832 Check if the Sentence Is Pangram (đủ 26 chữ cái) — cũng dùng `set`/bitmask.
5. LeetCode 2351 First Letter to Appear Twice — cùng kiểu theo dõi "đã thấy" bằng tập.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `in` trên chuỗi | thử từng chữ cái | `O(26n)` | `O(1)` | dễ nhất, `n` nhỏ |
| 2. `set` + `Z→A` | tra tập `O(1)` | `O(n)` | `O(1)` | **khuyến nghị khi nộp**, dừng sớm |
| 3. Hai mảng boolean | chỉ số bảng chữ cái | `O(n)` | `O(26)` | không dùng `set`, kiểm soát bộ nhớ |
| 4. Giao hai tập + `max` | đại số tập hợp | `O(n)` | `O(1)` | ngắn gọn, thể hiện ý tưởng "giao" |
| 5. Bitmask | 26 bit mỗi tập | `O(n)` | `O(1)` | nhanh, kỹ thuật bit |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print("Ab".swapcase(), {"a", "A"} & {"a"}, max([], default="-"), ord("c") - ord("a"))` in ra gì?

**Bài 2 (tính tay).** Với `s = "xXyYzZaB"`, chữ cái lớn nhất có cả hai dạng là gì? Còn `s = "aBcD"`?

**Bài 3 (điền TODO).** Hoàn thành hàm trả về chữ cái **nhỏ nhất** có cả hai dạng (chữ hoa) hoặc `""`.

In [24]:
def smallest_letter(s):
    # TODO: lấy giao của tập chữ thường và tập chữ hoa (đã lower), rồi min
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `count_both(s)` trả về **số chữ cái** xuất hiện ở cả hai dạng.

In [25]:
def count_both(s):
    # TODO: tự viết, gợi ý len(lowers & uppers)
    pass

**Bài 5 (mini challenge).** Viết `all_both(s)` trả về **chuỗi các chữ cái** (hoa) có cả hai dạng, sắp xếp tăng dần. Ví dụ `"arRAzFif"` → `"AFR"`.

In [26]:
def all_both(s):
    # TODO: sorted(...) và join
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"Ab".swapcase()="aB"`, giao là `{"a"}`, `max([], default="-")="-"`, `ord("c")-ord("a")=2`. Bài 2: `xXyYzZaB`: có `X/x`, `Y/y`, `Z/z` → `Z`. `aBcD`: mỗi chữ chỉ một dạng → `""`.
- **Mức 2 (vừa):** Bài 3: `min(lowers & uppers, default="").upper()`. Bài 4: kích thước giao hai tập.
- **Mức 3 (sâu):** Bài 5: `"".join(sorted(c.upper() for c in lowers & uppers))`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1 và 2
assert ("Ab".swapcase(), {"a", "A"} & {"a"}, max([], default="-"), ord("c") - ord("a")) == ("aB", {"a"}, "-", 2)
assert Solution().greatestLetter("xXyYzZaB") == "Z"
assert Solution().greatestLetter("aBcD") == ""

def both_sets(s):
    lowers = {c for c in s if c.islower()}
    uppers = {c.lower() for c in s if c.isupper()}
    return lowers & uppers

# Bài 3
def smallest_letter(s):
    return min(both_sets(s), default="").upper()

# Bài 4
def count_both(s):
    return len(both_sets(s))

# Bài 5
def all_both(s):
    return "".join(sorted(c.upper() for c in both_sets(s)))

assert smallest_letter("arRAzFif") == "A"
assert smallest_letter("lEeTcOdE") == "E"
assert smallest_letter("AbCdEfGhIjK") == ""
assert count_both("arRAzFif") == 3
assert count_both("abc") == 0
assert count_both("aAbBcC") == 3
assert all_both("arRAzFif") == "AFR"
assert all_both("zZ") == "Z"
assert all_both("xyz") == ""
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tập các ký tự đã thấy, duyệt từ `Z` xuống)

```python
import string

class Solution:
    def greatestLetter(self, s: str) -> str:
        seen = set(s)
        for upper in reversed(string.ascii_uppercase):
            if upper in seen and upper.lower() in seen:
                return upper
        return ""
```

**Checklist**

- [ ] Kết quả viết **hoa**?
- [ ] Kiểm tra **cả hai dạng** (`upper` và `lower`)?
- [ ] "Lớn nhất" theo bảng chữ cái (duyệt `Z→A` hoặc `max` trên cùng một dạng), không theo vị trí xuất hiện?
- [ ] Trường hợp không có chữ cái nào → trả `""` (không `ValueError`)?
- [ ] Đã thử `"zZ"` (lớn nhất) và `"aA"` (nhỏ nhất), chuỗi một dạng (`"ABCDEFG"`)?
- [ ] Không trộn hoa và thường khi so sánh bằng `max`?
- [ ] Signature đúng `greatestLetter(self, s: str) -> str` và có `from typing import List`?